# Non-prior, Prior, Last-year 돌린 후 FAILED까지 수기처리 후 돌릴 것.

In [3]:
# ─────────────────────────────────────────────────────────────────
# [셀 2] POST-PROCESSING: 매핑 + long 변환 + currency 환율 적용
#        + channel: US 매핑 / dummy 0 / PAID_NONPAID
#        + metric_col 키 분리 (J1~J7, J11)
#        + 개별 _stacked_separate.csv + 전체 union 저장
# ─────────────────────────────────────────────────────────────────
import re
from pathlib import Path
from datetime import datetime
import pandas as pd

MAPPING_CSV  = Path.cwd() / "tb_column_name_mapping.csv"
CURRENCY_CSV = Path.cwd() / "currency.csv"
EXPORTS_DIR  = Path.cwd() / "aa_exports"

# ── currency 로드 ─────────────────────────────────────────────────
cur_df = pd.read_csv(CURRENCY_CSV)
col_latest = cur_df.columns[2]
col_prior  = cur_df.columns[3]
year_latest = str(pd.to_datetime(col_latest).year)
year_prior  = str(pd.to_datetime(col_prior).year)
print(f"▶ 환율 컬럼: {col_latest}({year_latest}) / {col_prior}({year_prior})")

cur_map = {
    str(r["site_code"]).strip().lower(): {
        year_latest: r[col_latest],
        year_prior:  r[col_prior],
    }
    for _, r in cur_df.iterrows()
}

# ── channel 매핑 정의 ─────────────────────────────────────────────
us_mapping = {
    "US Channel":"Global Channel Mapping ",
    "Affiliate":"Affiliate Marketing",
    "Display":"Display",
    "Display Retargeting":"Display",
    "Other External Campaign":"Paid Others",
    "Other Paid Ecomm":"Paid Others",
    "Paid Search":"Paid Search",
    "Paid Search - eComm":"Paid Search",
    "PLA":"Pmax",
    "Social (Paid)":"Social Media Campaigns",
    "Vanity":"Paid Others",
    "Direct":"Direct",
    "Email - CRM":"Email",
    "Email - eComm":"Email",
    "Email - Upsell it":"Email",
    "Email (Retired)":"Email",
    "EPP":"EPP - US",
    "Natural Search":"Natural Search",
    "Other":"Other",
    "None":"Other",
    "Push Notifications":"Push",
    "Referring Domains":"Other",
    "Session Refresh":"Session Refresh",
    "Social (Free and Owned)":"Owned Social",
    "Social (Retired)":"Social Network Referrals",
    "Other External CampaignSegments":"Mobile Application",
    "Other External CampaignUS_Smartthings":"Mobile Application - Smartthings",
    "Other External CampaignUS_Samsung Members":"Mobile Application - Samsung Members",
    "SMS":"SMS",
    "Gen AI Search":"Gen AI (Organic)"
}

global_paid_mapping = {
    "Paid Search":"Paid",
    "Social Media Campaigns":"Paid",
    "Affiliate Marketing":"Paid",
    "Display":"Paid",
    "Other Campaigns":"Paid",
    "QR code (Paid)":"Paid",
    "Pmax":"Paid",
    "Demand Gen":"Paid",
    "Paid Others":"Paid",
    "Video":"Paid",
    "Gen AI (Paid)":"Paid",
    "Session Refresh":"Non-Paid",
    "Email":"Non-Paid",
    "Direct":"Non-Paid",
    "Natural Search":"Non-Paid",
    "Mobile Application":"Non-Paid",
    "Push":"Non-Paid",
    "Social Network Referrals":"Non-Paid",
    "Other":"Non-Paid",
    "SMS":"Non-Paid",
    "None":"Non-Paid",
    "Owned Social":"Non-Paid",
    "QR code (Owned)":"Non-Paid",
    "Samsung Web":"Non-Paid",
    "Gen AI (Organic)":"Non-Paid",
    "Owned Others":"Non-Paid"
}

# ── 최종 컬럼 정리 ────────────────────────────────────────────────
def finalize_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "PAID_NONPAID" not in df.columns:
        df["PAID_NONPAID"] = "-"
    df["TIER"]  = ""
    df["공란1"] = ""
    df["공란2"] = ""
    df["공란3"] = ""
    df["공란4"] = ""
    # KEY = SITE CODE(lower) + "_" + metric_col
    if "Site_Code" in df.columns and "metric_col" in df.columns:
        df["metric_col"] = df["Site_Code"].str.strip().str.lower() + "_" + df["metric_col"].astype(str)
    df = df.rename(columns={
        "Subsidiary":         "SUBS",
        "Country":            "COUNTRY",
        "Site_Code":          "SITE CODE",
        "J11":                "REPORT NO.",
        "J3":                 "DIVISION",
        "J4":                 "DATE",
        "J5":                 "DEVICE TYPE",
        "J6":                 "TYPE",
        "J7":                 "LOGIN/NON",
        "PAID_NONPAID":       "PAID/NONPAID",
        "value":              "ITEM",
        "metric_value_adj":   "VALUE",
        "metric_col":         "KEY",
        "metric_value_origin":"value_origin",
        "Start_Date":         "start_date",
        "End_Date":           "end_date",
    })
    final_cols = [
        "TIER", "SUBS", "COUNTRY", "SITE CODE",
        "REPORT NO.", "DIVISION", "DATE", "DEVICE TYPE", "TYPE", "LOGIN/NON",
        "PAID/NONPAID", "ITEM", "VALUE", "KEY",
        "공란1", "공란2", "공란3", "공란4",
        "value_origin", "start_date", "end_date"
    ]
    return df[[c for c in final_cols if c in df.columns]]


# ── 캠페인 키 분리 유틸 (260109v3.py 참조) ───────────────────────
report_no_mapping = {
    "1_1": "1_1~2. S.com Traffic by Division",
    "2_1": "2_1~4. Basic Traffic",
    "3_1": "3_1. Traffic by Channel (Internal)",
    "3_2": "3_2. Traffic by Channel (External)",
    "3_3": "3_3. Home KV & GNB to Campaign Page",
    "4_1": "4_1. Order Conversion with Login/Non_Login",
    "4_2": "4_2. Order Conversion with Login/Non_Login (Visit)",
    "5_1": "5_1. S.com Order Conversion",
    "6_1": "6_1. S.com Cross Sell Order (Multi Purchase)",
    "6_2": "6_2. S.com Cross Sell Order (Total)",
    "6_3": "6_3. Campaign Page Cross Sell Order",
    "7_1": "7_1~2. Order Conversion/Traffic by Channel"
}

def _split_by_number_pattern(parts):
    # metric_col이 X_Y_... 형태로 시작할 때 (예: 1_1_da_2025_scom_...)
    if len(parts) >= 2 and re.fullmatch(r"\d{1,2}", parts[0]) and re.fullmatch(r"\d{1,2}", parts[1]):
        return "", f"{parts[0]}_{parts[1]}", parts[2:]
    for i in (2, 3):
        if i + 1 < len(parts) and parts[i].isdigit() and parts[i+1].isdigit():
            return '_'.join(parts[:i]), f"{parts[i]}_{parts[i+1]}", parts[i+2:]
    return parts[0], f"{parts[1]}_{parts[2]}", parts[3:]

def _find_report_key(values):
    for v in values:
        if isinstance(v, str) and re.fullmatch(r"\d+_\d+", v):
            return v
    return ''

def split_metric_col(val: str) -> dict:
    """metric_col → J1~J7, J11"""
    parts = str(val).split('_')
    if len(parts) < 3:
        j1_to_j7 = (parts + [''] * 7)[:7]
        return {**{f"J{i+1}": j1_to_j7[i] for i in range(7)}, "J11": "", "metric_name": ""}

    first, second, remaining = _split_by_number_pattern(parts)
    processed = []
    i = 0
    while i < len(remaining):
        if remaining[i].isdigit() and remaining[i].startswith('202'):
            if i + 2 < len(remaining) and remaining[i+2].lower() == 'prior':
                processed.append(f"{remaining[i]}_{remaining[i+1]}_{remaining[i+2]}")
                i += 3
            elif i + 1 < len(remaining):
                processed.append(f"{remaining[i]}_{remaining[i+1]}")
                i += 2
            else:
                processed.append(remaining[i])
                i += 1
        else:
            processed.append(remaining[i])
            i += 1

    all_parts = [first, second] + processed
    j1_to_j7 = (all_parts + [''] * 7)[:7]
    metric_name = '_'.join(all_parts[7:]) if len(all_parts) > 7 else ''
    report_key = _find_report_key(j1_to_j7)
    return {**{f"J{i+1}": j1_to_j7[i] for i in range(7)}, "J11": report_no_mapping.get(report_key, ''), "metric_name": metric_name}

def _apply_j_cols(df: pd.DataFrame) -> pd.DataFrame:
    records = [split_metric_col(v) for v in df["metric_col"]]
    if not records:
        # df가 0행일 때: 컬럼만 추가하고 반환 (metric_name 누락 방지)
        extra_cols = [f"J{i+1}" for i in range(7)] + ["J11", "metric_name"]
        df = df.copy()
        for col in extra_cols:
            df[col] = pd.NA
        return df
    j_df = pd.DataFrame(records, index=df.index)
    return pd.concat([df, j_df], axis=1)

# ── 매핑 로드 ─────────────────────────────────────────────────────
mapping_df = pd.read_csv(MAPPING_CSV)
id_vars = ["Subsidiary", "Country", "Site_Code", "RSID",
           "Start_Date", "End_Date", "itemId", "value"]

ok_list, skip_list, no_vars_list = [], [], []
channel_frames = {}
all_frames = []  # union용

_all_csv = [f for f in EXPORTS_DIR.glob("*.csv") if "_long" not in f.name and "_stacked" not in f.name]
_TS_PAT = re.compile(r"_(\d{8}_\d{6})$")

def _get_ts(p):
    m = _TS_PAT.search(p.stem)
    return m.group(1) if m else ""

_latest_map = {}
for _f in _all_csv:
    _key = _TS_PAT.sub("", _f.stem)
    if _key not in _latest_map or _get_ts(_f) > _get_ts(_latest_map[_key]):
        _latest_map[_key] = _f

csv_files = sorted(_latest_map.values())
print(f"처리 대상: {len(csv_files)}개 파일 (tb_key별 최신 파일 기준)")
print()

for src_path in sorted(csv_files):
    stem   = src_path.stem
    tb_key = re.sub(r"_\d{8}_\d{6}$", "", stem)

    tb_mapping = mapping_df[mapping_df["tb"] == tb_key]
    if tb_mapping.empty:
        tb_mapping = mapping_df[mapping_df["tb"] == re.sub(r"_prior$", "", tb_key)]

    if tb_mapping.empty:
        skip_list.append(tb_key)
        continue

    rename_dict = dict(zip(tb_mapping["value_n"], tb_mapping["column"]))

    df = pd.read_csv(src_path, encoding="utf-8-sig")
    df = df.rename(columns=rename_dict)
    value_vars = [c for c in rename_dict.values() if c in df.columns]

    if not value_vars:
        no_vars_list.append(tb_key)
        continue

    for v in value_vars:
        df[v] = pd.to_numeric(
            df[v].astype(str).str.strip().str.replace("'", "", regex=False),
            errors="coerce"
        )

    df_long = df.melt(
        id_vars=[c for c in id_vars if c in df.columns],
        value_vars=value_vars,
        var_name="metric_col",
        value_name="metric_value_origin"
    )

    # ── currency 환율 적용 ────────────────────────────────────────
    df_long["_year"] = pd.to_datetime(df_long["Start_Date"]).dt.year.astype(str)
    df_long["_site"] = df_long["Site_Code"].str.strip().str.lower()

    def calc_adj(row):
        if "revenue" not in str(row["metric_col"]).lower():
            return row["metric_value_origin"]
        rates = cur_map.get(row["_site"])
        if not rates:
            return row["metric_value_origin"]
        rate = rates.get(row["_year"])
        if rate is None:
            return row["metric_value_origin"]
        try:
            return float(row["metric_value_origin"]) * float(rate)
        except (TypeError, ValueError):
            return row["metric_value_origin"]

    df_long["metric_value_adj"] = df_long.apply(calc_adj, axis=1)
    df_long = df_long.drop(columns=["_year", "_site"])

    df_long["metric_value_origin"] = pd.to_numeric(
        df_long["metric_value_origin"].astype(str).str.replace("'", "", regex=False),
        errors="coerce"
    )
    df_long["metric_value_adj"] = pd.to_numeric(
        df_long["metric_value_adj"].astype(str).str.replace("'", "", regex=False),
        errors="coerce"
    )

    # ── metric_col 키 분리 ────────────────────────────────────────
    df_long = _apply_j_cols(df_long)

    # ── channel 파일이면 수집, 아니면 바로 저장 ───────────────────
    is_channel = "channel" in tb_key.lower()
    if is_channel:
        channel_frames[tb_key] = df_long
    else:
        df_long["value"] = df_long["metric_name"]
        out_path = EXPORTS_DIR / f"{tb_key}_stacked_separate.csv"
        df_long.to_csv(out_path, index=False, encoding="utf-8-sig", float_format="%.6f")
        ok_list.append(tb_key)
        if re.search(r'_\d{1,2}_\d{1,2}(?:_|$)', tb_key):
            all_frames.append(df_long)

# ── channel 후처리 (루프 종료 후) ────────────────────────────────
if channel_frames:
    print(f"\n▶ channel 후처리: {list(channel_frames.keys())}")

    for tb_key, df_long in channel_frames.items():

        # 1. US value 치환
        is_us = df_long["Site_Code"].str.strip().str.lower() == "us"
        df_long.loc[is_us, "value"] = (
            df_long.loc[is_us, "value"].map(us_mapping).fillna(df_long.loc[is_us, "value"])
        )

        # 2. dummy 0 삽입
        all_sites   = df_long["Site_Code"].unique()
        all_values  = df_long["value"].unique()
        metric_cols = df_long["metric_col"].unique()

        meta_cols = [c for c in ["Subsidiary", "Country", "RSID"] if c in df_long.columns]
        meta_df = (
            df_long.groupby("Site_Code")[meta_cols]
            .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
            .reset_index()
        )
        date_cols = [c for c in ["Start_Date", "End_Date", "itemId"] if c in df_long.columns]
        date_meta = (
            df_long.groupby("Site_Code")[date_cols]
            .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
            .reset_index()
        )

        cross = pd.MultiIndex.from_product(
            [all_sites, all_values, metric_cols],
            names=["Site_Code", "value", "metric_col"]
        ).to_frame(index=False)

        existing = df_long[["Site_Code", "value", "metric_col"]].drop_duplicates()
        existing["_exists"] = True

        merged = cross.merge(existing, on=["Site_Code", "value", "metric_col"], how="left")
        dummy_rows = merged[merged["_exists"].isna()].drop(columns=["_exists"])

        if not dummy_rows.empty:
            dummy_rows = dummy_rows.merge(meta_df, on="Site_Code", how="left")
            dummy_rows = dummy_rows.merge(date_meta, on="Site_Code", how="left")
            dummy_rows["metric_value_origin"] = 0.0
            dummy_rows["metric_value_adj"]    = 0.0
            dummy_rows = _apply_j_cols(dummy_rows)
            df_long = pd.concat([df_long, dummy_rows], ignore_index=True)
            print(f"  {tb_key} → dummy {len(dummy_rows)}행 삽입")

        # 3. PAID_NONPAID 컬럼 추가 (value 왼쪽)
        df_long["PAID_NONPAID"] = df_long["value"].map(global_paid_mapping)
        val_idx = df_long.columns.get_loc("value")
        cols = list(df_long.columns)
        cols.remove("PAID_NONPAID")
        cols.insert(val_idx, "PAID_NONPAID")
        df_long = df_long[cols]
        # channel KEY: value에 연도(20XX) 없을 때만 metric_col 뒤에 ITEM 추가
        _has_year = df_long["value"].astype(str).str.contains(r'\b20\d{2}\b', regex=True, na=False)
        df_long["metric_col"] = df_long["metric_col"].astype(str).where(
            _has_year,
            df_long["metric_col"].astype(str) + df_long["value"].astype(str)
        )
        # year 행: ITEM도 metric_name으로 교체 (날짜값 대신 채널 카테고리 표시)
        df_long.loc[_has_year, "value"] = df_long.loc[_has_year, "metric_name"].str.rstrip("_")

        out_path = EXPORTS_DIR / f"{tb_key}_stacked_separate.csv"
      #   df_long.to_csv(out_path, index=False, encoding="utf-8-sig", float_format="%.6f")
        finalize_df(df_long).to_csv(out_path, index=False, encoding="utf-8-sig", float_format="%.6f")
        ok_list.append(tb_key)
        if re.search(r'_\d{1,2}_\d{1,2}(?:_|$)', tb_key):
            all_frames.append(df_long)

# ── union 생성 ────────────────────────────────────────────────────
if all_frames:
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    union_path = EXPORTS_DIR / f"union_{ts}.csv"
    union_df = pd.concat(all_frames, ignore_index=True)
    if "PAID_NONPAID" in union_df.columns:
        union_df["PAID_NONPAID"] = union_df["PAID_NONPAID"].fillna("-")
    # US dummy: non-US에 _app/_android/_ios 키 있고 US에 없으면 0행 삽입
    _app_pat = r"_(?:app|android|ios)"
    _non_us  = union_df["Site_Code"].str.strip().str.upper() != "US"
    _us_mask = union_df["Site_Code"].str.strip().str.upper() == "US"
    _app_mc  = union_df.loc[
        _non_us & union_df["metric_col"].str.lower().str.contains(_app_pat, regex=True, na=False),
        "metric_col"
    ].unique()
    _us_exist = set(union_df.loc[_us_mask, "metric_col"].unique())
    _missing  = [mc for mc in _app_mc if mc not in _us_exist]
    if _missing:
        _us_rows = union_df[_us_mask]
        if _us_rows.empty:
            print("[WARN] No US data -> skip _app/_android/_ios dummy insert")
        else:
            _tmpl = _us_rows.iloc[0].to_dict()
            _dummy_list = []
            for _mc in _missing:
                _ref = union_df.loc[_non_us & (union_df["metric_col"] == _mc)]
                _row = {col: None for col in union_df.columns}
                _row.update(_tmpl)
                _row["metric_col"] = _mc
                _row["metric_value_origin"] = 0.0
                _row["metric_value_adj"]    = 0.0
                if not _ref.empty:
                    for _dc in ["Start_Date", "End_Date"]:
                        if _dc in _ref.columns:
                            _row[_dc] = _ref.iloc[0][_dc]
                _j = split_metric_col(_mc)
                _row.update(_j)
                _row["value"] = _j.get("metric_name", "")
                if "PAID_NONPAID" in union_df.columns:
                    _row["PAID_NONPAID"] = "-"
                _dummy_list.append(_row)
            _us_dummy_df = pd.DataFrame(_dummy_list, columns=union_df.columns)
            union_df = pd.concat([union_df, _us_dummy_df], ignore_index=True)
            print(f"US _app/_android/_ios dummy {len(_dummy_list)} rows inserted")
  #   union_df.to_csv(union_path, index=False, encoding="utf-8-sig", float_format="%.6f")
    finalize_df(union_df).to_csv(union_path, index=False, encoding="utf-8-sig", float_format="%.6f")
    print(f"\n▶ union 저장: {union_path} ({len(union_df):,}행)")

# ── 결과 요약 ─────────────────────────────────────────────────────
print(f"\n✅ 완료: {len(ok_list)}개")
for t in ok_list: print(f"  - {t}")
print(f"\n⚠️  매핑 없어서 skip: {len(skip_list)}개")
for t in skip_list: print(f"  - {t}")
print(f"\n⚠️  매핑컬럼 불일치 skip: {len(no_vars_list)}개")
for t in no_vars_list: print(f"  - {t}")

▶ 환율 컬럼: 2026-02-28(2026) / 2025-02-28(2025)
▶ 처리 대상: 125개 파일


▶ channel 후처리: ['25_ny_3_1_channel_internal', '25_ny_3_2_channel_traffic_external', '25_ny_7_1_order_by_channel_cmp', '26_ny_3_1_channel_internal', '26_ny_3_2_channel_external_cmp', '26_ny_3_2_channel_external_scom', '26_ny_3_2_channel_external_scom_prior', '26_ny_7_1_order_by_channel_cmp', '26_ny_7_1_order_by_channel_scom', '26_ny_7_1_order_by_channel_scom_prior', 'us_26_ny_3_1_channel_internal', 'us_26_ny_3_2_channel_external_cmp', 'us_26_ny_3_2_channel_external_scom', 'us_26_ny_3_2_channel_external_scom_prior', 'us_26_ny_7_1_order_by_channel_cmp', 'us_26_ny_7_1_order_by_channel_scom', 'us_26_ny_7_1_order_by_channel_scom_prior']
  25_ny_3_2_channel_traffic_external → dummy 522행 삽입
  25_ny_7_1_order_by_channel_cmp → dummy 537행 삽입
  26_ny_3_2_channel_external_cmp → dummy 2814행 삽입
  26_ny_3_2_channel_external_scom → dummy 832행 삽입
  26_ny_3_2_channel_external_scom_prior → dummy 852행 삽입
  26_ny_7_1_order_by_channel_cmp → dumm